# Separating Input Optimizer - Quick Demo

This notebook demonstrates how to use the **Separating Input Optimizer** to find control inputs that maximize separation between different fault scenarios.

## What is a Separating Input?

A **separating input** is a control signal that minimizes the overlap between interval reachable sets for different fault scenarios. This makes it easier to distinguish between faults by observing the system's behavior.

### Key Idea
- Apply control input `u`
- Propagate system under different fault scenarios
- Minimize overlap between resulting reachable sets
- Result: Control that maximally separates fault scenarios

## Setup

In [ ]:
import jax.numpy as jnp
import numpy as np
import immrax as irx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle

from faulty_planar_multirotor import FaultyPlanarMultirotor
from separating_input_optimizer import (
    SeparatingInputOptimizer,
    optimize_separating_input_multistart,
)

print("✓ Imports successful")

## Example 1: Two Fault Scenarios

We'll compare two scenarios:
- **Nominal**: 100% thrust effectiveness
- **Fault**: 50% thrust effectiveness (broken rotor)

Goal: Find a control input that maximally separates their reachable sets.

### Step 1: Define the System

In [ ]:
# Planar multirotor system with fault models
sys = FaultyPlanarMultirotor()

print(f"System dimensions:")
print(f"  States: {sys.xlen} (px, py, vx, vy, theta)")
print(f"  Inputs: {sys.ulen} (thrust, angular_accel)")
print(f"  Parameters: {sys.plen} (thrust_eff, angular_eff)")

### Step 2: Define Initial Conditions and Uncertainties

In [ ]:
# Initial state interval: start near origin with some uncertainty
x0_center = jnp.array([0.0, 0.0, 0.0, 0.0, 0.0])  # px, py, vx, vy, theta
x0_perturbation = jnp.array([0.3, 0.3, 0.2, 0.2, 0.1])
x0 = irx.icentpert(x0_center, x0_perturbation)

# No disturbance for simplicity
w = irx.icentpert(jnp.zeros(1), jnp.zeros(1))

print("Initial state interval:")
print(f"  px: [{float(x0.lower[0]):.2f}, {float(x0.upper[0]):.2f}]")
print(f"  py: [{float(x0.lower[1]):.2f}, {float(x0.upper[1]):.2f}]")
print(f"  vx: [{float(x0.lower[2]):.2f}, {float(x0.upper[2]):.2f}]")
print(f"  vy: [{float(x0.lower[3]):.2f}, {float(x0.upper[3]):.2f}]")
print(f"  theta: [{float(x0.lower[4]):.2f}, {float(x0.upper[4]):.2f}]")

### Step 3: Define Fault Scenarios

In [ ]:
# Nominal parameters: 100% thrust, 100% angular control
p_nominal = irx.icentpert(jnp.array([1.0, 1.0]), jnp.zeros(2))

# Fault parameters: 50% thrust (broken rotor), 100% angular control
p_fault = irx.icentpert(jnp.array([0.5, 1.0]), jnp.zeros(2))

print("Fault scenarios:")
print(f"  Nominal: thrust=100%, angular=100%")
print(f"  Fault:   thrust=50%, angular=100%")

### Step 4: Create the Optimizer

In [ ]:
optimizer = SeparatingInputOptimizer(
    system=sys,
    fault_parameters=[p_nominal, p_fault],
    x0_interval=x0,
    w_interval=w,
    dt=0.02,  # 20ms timestep
    num_steps=10,  # Propagate 10 steps (0.2s total)
    state_slice=slice(0, 2),  # Only consider position (px, py) for overlap
)

print("✓ Optimizer created")
print(f"  Propagation time: {optimizer.dt * optimizer.num_steps:.3f}s")
print(f"  Monitoring states: position (px, py)")

### Step 5: Evaluate Baseline (Hover Input)

In [ ]:
# Try hover input as baseline: u = [g, 0] (counteract gravity, no rotation)
u_hover = jnp.array([9.81, 0.0])

loss_hover, intervals_hover = optimizer.evaluate(u_hover)

print("Baseline (hover input):")
print(f"  Control: u = [{float(u_hover[0]):.2f}, {float(u_hover[1]):.2f}]")
print(f"  Overlap: {float(loss_hover):.6f}")
print(f"\nFinal position intervals:")
print(f"  Nominal: px=[{float(intervals_hover[0].lower[0]):.3f}, {float(intervals_hover[0].upper[0]):.3f}], "
      f"py=[{float(intervals_hover[0].lower[1]):.3f}, {float(intervals_hover[0].upper[1]):.3f}]")
print(f"  Fault:   px=[{float(intervals_hover[1].lower[0]):.3f}, {float(intervals_hover[1].upper[0]):.3f}], "
      f"py=[{float(intervals_hover[1].lower[1]):.3f}, {float(intervals_hover[1].upper[1]):.3f}]")

### Step 6: Optimize Separating Input

In [ ]:
# Run gradient descent optimization
u_opt, loss_opt = optimizer.optimize(
    u_initial=u_hover,  # Start from hover
    learning_rate=1e-1,  # Learning rate
    num_iterations=100,  # Number of gradient steps
    verbose=False,
)

print("Optimized separating input:")
print(f"  Control: u = [{float(u_opt[0]):.3f}, {float(u_opt[1]):.3f}]")
print(f"  Overlap: {float(loss_opt):.6f}")

if float(loss_hover) > 1e-8:
    improvement = (1 - float(loss_opt)/float(loss_hover)) * 100
    print(f"  Improvement: {improvement:.1f}% reduction in overlap")
else:
    print(f"  Note: Baseline already had zero overlap")

### Step 7: Evaluate Optimized Input

In [ ]:
# Evaluate the optimized control
loss_opt_check, intervals_opt = optimizer.evaluate(u_opt)

print("Final intervals with optimized control:")
print(f"  Nominal: px=[{float(intervals_opt[0].lower[0]):.3f}, {float(intervals_opt[0].upper[0]):.3f}], "
      f"py=[{float(intervals_opt[0].lower[1]):.3f}, {float(intervals_opt[0].upper[1]):.3f}]")
print(f"  Fault:   px=[{float(intervals_opt[1].lower[0]):.3f}, {float(intervals_opt[1].upper[0]):.3f}], "
      f"py=[{float(intervals_opt[1].lower[1]):.3f}, {float(intervals_opt[1].upper[1]):.3f}]")

### Visualization: Compare Baseline vs. Optimized

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Helper function to plot intervals
def plot_intervals(ax, intervals, title):
    colors = ['blue', 'red']
    labels = ['Nominal', 'Fault']
    alphas = [0.3, 0.3]
    
    for i, (interval, color, label, alpha) in enumerate(zip(intervals, colors, labels, alphas)):
        px_low, py_low = float(interval.lower[0]), float(interval.lower[1])
        px_high, py_high = float(interval.upper[0]), float(interval.upper[1])
        width = px_high - px_low
        height = py_high - py_low
        
        rect = Rectangle((px_low, py_low), width, height, 
                        linewidth=2, edgecolor=color, 
                        facecolor=color, alpha=alpha, label=label)
        ax.add_patch(rect)
    
    ax.set_xlabel('Position X (m)', fontsize=12)
    ax.set_ylabel('Position Y (m)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)
    ax.axis('equal')
    ax.set_aspect('equal', adjustable='box')

# Plot baseline
plot_intervals(axes[0], intervals_hover, 
              f'Baseline (Hover)\nOverlap = {float(loss_hover):.4f}')

# Plot optimized
plot_intervals(axes[1], intervals_opt, 
              f'Optimized\nOverlap = {float(loss_opt):.4f}')

plt.tight_layout()
plt.show()

print(f"\n📊 Visualization:")
print(f"  Blue = Nominal scenario reachable set")
print(f"  Red = Fault scenario reachable set")
print(f"  Purple overlap = Region where scenarios are indistinguishable")

## Example 2: Three Fault Scenarios with Multi-Start

Now let's try a more challenging problem with three fault scenarios and use multi-start optimization for better results.

In [ ]:
# Define three scenarios with different thrust levels
p_100 = irx.icentpert(jnp.array([1.0, 1.0]), jnp.zeros(2))  # 100% thrust
p_70 = irx.icentpert(jnp.array([0.7, 1.0]), jnp.zeros(2))   # 70% thrust
p_40 = irx.icentpert(jnp.array([0.4, 1.0]), jnp.zeros(2))   # 40% thrust

print("Three fault scenarios:")
print("  Scenario 1: 100% thrust")
print("  Scenario 2: 70% thrust")
print("  Scenario 3: 40% thrust")
print("\nGoal: Find control that separates ALL three scenarios")

In [ ]:
# Use multi-start optimization for global search
print("Running multi-start optimization (3 random restarts)...")

u_opt_3, loss_opt_3 = optimize_separating_input_multistart(
    system=sys,
    fault_parameters=[p_100, p_70, p_40],
    x0_interval=x0,
    w_interval=w,
    dt=0.02,
    num_steps=10,
    num_restarts=3,  # Try 3 different random initializations
    learning_rate=1e-1,
    num_iterations=80,
    state_slice=slice(0, 2),
    random_key=42,
)

print(f"\nOptimal control: u = [{float(u_opt_3[0]):.3f}, {float(u_opt_3[1]):.3f}]")
print(f"Total pairwise overlap: {float(loss_opt_3):.6f}")
print(f"(This is sum of overlaps: 1-2, 1-3, 2-3)")

In [ ]:
# Evaluate to get final intervals
optimizer_3 = SeparatingInputOptimizer(
    system=sys,
    fault_parameters=[p_100, p_70, p_40],
    x0_interval=x0,
    w_interval=w,
    dt=0.02,
    num_steps=10,
    state_slice=slice(0, 2),
)
_, intervals_3 = optimizer_3.evaluate(u_opt_3)

print("Final position intervals:")
labels = ['100% thrust', '70% thrust', '40% thrust']
for interval, label in zip(intervals_3, labels):
    print(f"  {label}: px=[{float(interval.lower[0]):.3f}, {float(interval.upper[0]):.3f}], "
          f"py=[{float(interval.lower[1]):.3f}, {float(interval.upper[1]):.3f}]")

In [ ]:
# Visualize three scenarios
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

colors = ['blue', 'green', 'red']
labels = ['100% thrust', '70% thrust', '40% thrust']
alphas = [0.25, 0.25, 0.25]

for interval, color, label, alpha in zip(intervals_3, colors, labels, alphas):
    px_low, py_low = float(interval.lower[0]), float(interval.lower[1])
    px_high, py_high = float(interval.upper[0]), float(interval.upper[1])
    width = px_high - px_low
    height = py_high - py_low
    
    rect = Rectangle((px_low, py_low), width, height, 
                    linewidth=2, edgecolor=color, 
                    facecolor=color, alpha=alpha, label=label)
    ax.add_patch(rect)

ax.set_xlabel('Position X (m)', fontsize=12)
ax.set_ylabel('Position Y (m)', fontsize=12)
ax.set_title(f'Three Fault Scenarios\nTotal Pairwise Overlap = {float(loss_opt_3):.4f}', 
            fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='best')
ax.axis('equal')
ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.show()

print("\n📊 Three scenarios visualized!")
print("   Smaller overlap = better separation = easier fault identification")

## Example 3: Uncertain Fault Parameters

In practice, fault severity may be uncertain. Let's handle that case.

In [ ]:
# Nominal: certain
p_nominal_certain = irx.icentpert(jnp.array([1.0, 1.0]), jnp.zeros(2))

# Fault: uncertain thrust effectiveness in [30%, 70%]
p_fault_uncertain = irx.icentpert(
    jnp.array([0.5, 1.0]),   # Center: 50% thrust
    jnp.array([0.2, 0.0]),   # Uncertainty: ±20% (so [30%, 70%])
)

print("Uncertain fault scenario:")
print(f"  Nominal: thrust = 100% (certain)")
print(f"  Fault:   thrust = 50% ± 20% (uncertain, range [30%, 70%])")
print(f"\nParameter interval for fault:")
print(f"  [{float(p_fault_uncertain.lower[0]):.2f}, {float(p_fault_uncertain.upper[0]):.2f}]")

In [ ]:
optimizer_uncertain = SeparatingInputOptimizer(
    system=sys,
    fault_parameters=[p_nominal_certain, p_fault_uncertain],
    x0_interval=x0,
    w_interval=w,
    dt=0.02,
    num_steps=10,
    state_slice=slice(0, 2),
)

u_opt_unc, loss_opt_unc = optimizer_uncertain.optimize(
    u_initial=jnp.array([9.81, 0.0]),
    learning_rate=1e-1,
    num_iterations=100,
)

print(f"Optimal control: u = [{float(u_opt_unc[0]):.3f}, {float(u_opt_unc[1]):.3f}]")
print(f"Overlap: {float(loss_opt_unc):.6f}")
print(f"\nNote: The optimizer accounts for the worst-case parameter uncertainty!")

## Summary

### What We Learned

1. **Basic Usage**: Create optimizer, define fault scenarios, optimize
2. **Two Scenarios**: Nominal vs. fault with gradient descent
3. **Multiple Scenarios**: Use multi-start for 3+ scenarios
4. **Uncertain Parameters**: Handle parameter uncertainty in faults

### Key Takeaways

- ✅ Automatic differentiation via JAX makes optimization easy
- ✅ JIT compilation provides fast execution
- ✅ Multi-start helps find global optima
- ✅ Works with uncertain fault parameters
- ✅ Flexible: choose which states to monitor (position, velocity, etc.)

### Applications

- **Fault Diagnosis**: Design inputs for active fault detection
- **System Identification**: Excite different system modes
- **Robust Control**: Design inputs that reveal system uncertainties
- **Safety Verification**: Test system under various fault conditions

## Next Steps

Try these extensions:

1. **Different Systems**: Apply to other faulty systems (car, aircraft, etc.)
2. **Time-Varying Inputs**: Optimize sequences of controls (see `calculate_optimal_open_loop_u.py`)
3. **More States**: Monitor velocity, angle, or all states simultaneously
4. **Sensor Faults**: Include sensor drift/bias in fault scenarios
5. **Constraints**: Add box constraints on control inputs

See `README_separating_input.md` for full documentation!